In [1]:
import pandas as pd
import re
from fuzzywuzzy import fuzz



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

# Fonction pour nettoyer les noms des clubs

In [2]:

def clean_name(name):
   
    suffixes = r'\s*(FC|Club|SFC|SPFC|RET|Bait Jair|Sports Club|Sporting|Mohammedein FC)$'
    cleaned = re.sub(suffixes, '', str(name), flags=re.IGNORECASE)
    return cleaned.lower().strip()


# Fonction pour regroupement des noms d'équipes similaires (Exemple Al-Nassr FC, Nassr	→	Al-Nassr FC)

In [3]:
def standardize_team_names(df):
    # On récupère tous les noms d'équipes
    all_team_names = []
    for col in ['provider_1_team_name', 'provider_2_team_name', 'provider_3_team_name']:
        for name in df[col].dropna().unique():
            if name not in all_team_names:  # Évite les doublons
                all_team_names.append(name)
    
    # On prépare une liste pour stocker nos groupes d'équipes
    clusters = []
    
    SEUIL_SIMILARITE = 85 
    
    # On va traiter chaque nom un par un
    for name in all_team_names:
        trouve = False
        
        # On nettoie le nom  
        cleaned_name = clean_name(name)
        
        # On cherche si ça matche avec un groupe existant
        for cluster in clusters:
            # On prend le premier nom du groupe comme référence
            ref_name = cluster[0]
            cleaned_ref = clean_name(ref_name)
            
            # Calcul de la similarité avec fuzzy
            score = fuzz.partial_ratio(cleaned_name, cleaned_ref)
            
            if score > SEUIL_SIMILARITE:
                cluster.append(name)
                trouve = True
                break
        
        # Si on a rien trouvé, on crée un nouveau groupe
        if not trouve:
            clusters.append([name])
    
    # Maintenant on crée le mapping pour la standardisation
    name_mapping = {}
    for cluster in clusters:
        # On choisit le nom le plus long comme standard souvent le plus complet
        standard = max(cluster, key=len)
        
        # On ajoute chaque variante au mapping
        for variant in cluster:
            name_mapping[variant] = standard
    
    return name_mapping

In [4]:

def main():
    df = pd.read_excel('data/file_ASM.xlsx', sheet_name='club_names_ids')
    
    name_mapping = standardize_team_names(df)
    print(name_mapping)
    print("____________________________________________________________________")
    # Appliquer le mapping
    df['standard_team_name'] = df['provider_1_team_name'].map(name_mapping)
    
    # Sauvegarder le résultat
    df.to_excel('cleaned_teams_BM.xlsx', index=False)
    print("Standardisation terminée. Résultats sauvegardés dans 'cleaned_teams_BM.xlsx'")

if __name__ == "__main__":
    main()

{'Al-Nassr FC': 'Al-Nassr FC', 'Al-Nassr': 'Al-Nassr FC', 'Nassr': 'Al-Nassr FC', 'Al-Ittihad Club': 'Al-Ittihad Club', 'Al-Ittihad': 'Al-Ittihad Club', 'Ittihad': 'Al-Ittihad Club', 'Al-Shabab': 'Al-Shabab', 'Shabab': 'Al-Shabab', 'Al-Hilal SFC': 'Al-Hilal SFC', 'Al-Hilal': 'Al-Hilal SFC', 'Hilal': 'Al-Hilal SFC', 'Al-Taawoun FC': 'Taawoun Sports Club', 'Al-Taawoun RET': 'Taawoun Sports Club', 'Taawoun Sports Club': 'Taawoun Sports Club', 'Al-Fateh': 'Fateh Sporting', 'Fateh Sporting': 'Fateh Sporting', 'Damac FC': 'Damac Sporting FC', 'Damac': 'Damac Sporting FC', 'Damac Sporting FC': 'Damac Sporting FC', 'Al-Tai': 'Al-Tai', 'Tai FC': 'Al-Tai', 'Abha': 'Abha FC', 'Abha FC': 'Abha FC', 'Al-Raed': 'Al-Raed', 'Raed FC': 'Al-Raed', 'Al-Fayha': 'Al-Fayha', 'Fayha FC': 'Al-Fayha', 'Ettifaq FC': 'Ettifaq FC', 'Al-Ettifaq': 'Ettifaq FC', 'Al-Wehda FC': 'Al-Wehda SPFC', 'Al-Wehda SPFC': 'Al-Wehda SPFC', 'Wehda FC': 'Al-Wehda SPFC', 'Khaleej': 'Al-Khaleej', 'Al-Khaleej': 'Al-Khaleej', 'Khaleej